In [ ]:
!pip install "psycopg[binary]"

In [ ]:
print("Hello database")


import os
from dotenv import load_dotenv

load_dotenv()



# Connect to DataBase

In [ ]:
DB_HOST = "localhost"
DB_PORT = 5433
DB_NAME = "resumedb"
DB_USER = "ahmed"
DB_PASSWORD = os.getenv("DB_PASSWORD")

if not DB_PASSWORD:
    raise RuntimeError("DB_PASSWORD is not configured.")

import psycopg

cursor = None

conn = psycopg.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)


In [ ]:
cursor = conn.cursor()
cursor

# Get Response


In [ ]:
conn.rollback()

cursor.execute("SELECT current_database(), current_schema()")
print(cursor.fetchone())

In [ ]:
conn.rollback()

cursor.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_name IN ('documents', 'document_chunks')
""")

print(cursor.fetchall())

In [ ]:
conn.rollback()

try:
    cursor.execute("SELECT * FROM documents")
    print(cursor.fetchall())
except Exception as e:
    print(type(e).__name__, e)
    conn.rollback()

In [ ]:
conn.rollback

In [ ]:

res = cursor.execute("SELECT 1")
res.fetchone()


In [ ]:
res = cursor.execute("SELECT * FROM documents")
res.fetchall()

for raw in res.fetchall():
    print(raw)

# First Way Read Data (Parse Data by pymupdf)


In [ ]:
%pip install pymupdf

In [ ]:
import pymupdf 

pdf_path = r"Data/sample.pdf"

doc = pymupdf.open(pdf_path)
print(f"Number of pages: {len(doc)}")

In [ ]:
print(doc.metadata)

In [ ]:
page = doc[0]

text = page.get_text("text")

print(text)

In [ ]:
pages = []

for page_number , page in enumerate(doc , start=1):
    text = page.get_text("text")
    pages.append(
        {
            "page_number": page_number,
            "text" : text
        }
    )
    #print(f"page_number: {page_number} , text : {text[:500]}")
pages

In [ ]:
page = doc[0]

text = page.get_text("text", sort=True)

print(text)

# Second Way : Read Data from LIAMAIndex >> liamaparse

# Connect to service

In [ ]:
%pip uninstall -y llama-parse llama-cloud-services
%pip install -U llama-cloud

In [ ]:

import os

from llama_cloud import LlamaCloud
from dotenv import load_dotenv


load_dotenv()

api_key = os.environ.get("LLAMA_CLOUD_API_KEY")

if not api_key: 
    raise ValueError("LLAMA_CLOUD_API_KEY environment variable is not set")
client = LlamaCloud(api_key=api_key)

# Upload File

In [ ]:
from pathlib import Path

data_folder = Path("Data")

pdf_files = list(data_folder.glob("*.pdf"))
uploaded_files = []

if not pdf_files:
    print("No PDF files found in the Data folder.")

for pdf_path in pdf_files:
    with pdf_path.open("rb") as pdf_file:
        uploaded_file = client.files.create(
            file=pdf_file,
            purpose="parse"
        )

    uploaded_files.append(uploaded_file)
    print(f"Uploaded successfully: {pdf_path.name}")



# Analyze file

In [ ]:
results = {}

for uploaded_file in uploaded_files:
    result = client.parsing.parse(
        file_id=uploaded_file.id,
        tier="agentic",
        version="latest",
        expand=["markdown"]
    )

    results[uploaded_file.id] = result

    print(f"Successfully parsed file ID: {uploaded_file.id}")
    

In [ ]:
result.markdown.pages

# Extract and Show pages

In [ ]:
llama_pages=[]

for page in result.markdown.pages:
    if page.success:
        llama_pages.append(
            {
            "page": page.page_number,
            "text": page.markdown
            }
        )

llama_text = "\n\n".join(
    page["text"] for page in llama_pages
)

print(llama_text)